# 08 Beta Diversity - Settlement

### Load inputs and match samples

Load the 16-mer and whole ASV Bray–Curtis distance matrices and align the metadata to their sample IDs. Count settlement labels, distinguish missing labels from sample IDs absent from the metadata, and subset the matrix to Rural, Suburban, and Urban samples. We expect 150 samples in the full matrix and 145 in the subset.

In [7]:
import os
import pandas as pd
import qiime2 as q2
from skbio import OrdinationResults
from skbio.stats.distance import permanova
from qiime2 import Visualization, Artifact, Metadata
from seaborn import scatterplot

%matplotlib inline

In [35]:
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"

out_dir = "../data/processed_ignored/08_beta_diversity_settlement"
vis_dir = "../data/processed/08_beta_diversity_settlement"

kmer16_settlement_distance_path = f"{out_dir}/kmer16_bray_curtis_settlement.qza"
asv_settlement_distance_path = f"{out_dir}/asv_bray_curtis_settlement.qza"

settlement_metadata_path = f"{out_dir}/settlement_metadata.tsv"

# Overall PERMANOVAs:
kmer16_permanova_path = f"{vis_dir}/kmer16_bray_curtis_settlement_permanova.qzv"
asv_permanova_path = f"{vis_dir}/asv_bray_curtis_settlement_permanova.qzv"

# Pairwise PERMANOVAs:
kmer16_pairwise_permanova_path = f"{vis_dir}/kmer16_bray_curtis_settlement_pairwise_permanova.qzv"
asv_pairwise_permanova_path = f"{vis_dir}/asv_bray_curtis_settlement_pairwise_permanova.qzv"

metadata = q2.Metadata.load(metadata_path).to_dataframe()

! mkdir -p "$out_dir" "$vis_dir"

### Prepare inputs for the settlement tests

Keep samples labeled Rural, Suburban, or Urban. Save the filtered 16-mer and whole ASV Bray–Curtis matrices and their matching settlement metadata for QIIME 2.

In [36]:
kmer16_distance = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
settlement = metadata["Settlement_Type"].reindex(kmer16_distance.ids).str.strip()

keep = settlement.isin(["Rural", "Suburban", "Urban"])
settlement_metadata = settlement[keep].to_frame()
kmer16_settlement_distance = kmer16_distance.filter(settlement_metadata.index.tolist())

q2.Artifact.import_data("DistanceMatrix", kmer16_settlement_distance).save(
    kmer16_settlement_distance_path
)
q2.Metadata(settlement_metadata).save(settlement_metadata_path)

'../data/processed_ignored/08_beta_diversity_settlement/settlement_metadata.tsv'

### PERMANOVA by settlement type

Use QIIME 2 to test whether fungal community composition differs among Rural, Suburban, and Urban samples. Run one overall test with 999 permutations using the filtered 16-mer Bray–Curtis distance matrix.

In [21]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$kmer16_settlement_distance_path" \
    --m-metadata-file "$settlement_metadata_path" \
    --m-metadata-column Settlement_Type \
    --o-visualization "$kmer16_permanova_path"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/kmer16_bray_curtis_settlement_permanova.qzv


In [ ]:
Visualization.load(kmer16_permanova_path)

#### Results
Using 16-mer Bray–Curtis distances, the overall PERMANOVA found no statistically significant association between settlement type and fungal community composition (n = 145, pseudo-F = 1.496, p = 0.100, 999 permutations).

### Prepare the whole-ASV Bray–Curtis matrix

Keep the same sample IDs used in the 16-mer PERMANOVA and save the filtered whole-ASV matrix.

In [37]:
asv_distance = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)
asv_settlement_distance = asv_distance.filter(settlement_metadata.index.tolist())

q2.Artifact.import_data("DistanceMatrix", asv_settlement_distance).save(
    asv_settlement_distance_path
)

print("Samples retained:", len(asv_settlement_distance.ids))

Samples retained: 145


### PERMANOVA on whole-ASV Bray–Curtis distances

Repeat the overall settlement test on the same 145 samples using whole-ASV Bray–Curtis distances. This checks whether the conclusion depends on how the ITS1 sequences were represented.

In [25]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$asv_settlement_distance_path" \
    --m-metadata-file "$settlement_metadata_path" \
    --m-metadata-column Settlement_Type \
    --o-visualization "$asv_permanova_path"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/asv_bray_curtis_settlement_permanova.qzv


In [ ]:
Visualization.load(asv_permanova_path)

### Compare the settlement PERMANOVA results

| Bray–Curtis feature representation | Samples | Pseudo-F | p-value |
|---|---:|---:|---:|
| 16-mers | 145 | 1.496 | 0.100 |
| Whole ASVs | 145 | 1.671 | **0.027** |

#### Results
At α = 0.05, the overall PERMANOVA detected an association between settlement type and fungal community composition using whole-ASV Bray–Curtis distances (pseudo-F = 1.671, p = 0.027). The primary 16-mer Bray–Curtis analysis did not reach that threshold (pseudo-F = 1.496, p = 0.100). Both tests used the same 145 samples, so the significance conclusion depended on how the ITS1 features were represented.

### Pairwise PERMANOVA on 16-mer Bray–Curtis distances

Compare Rural, Suburban, and Urban samples two groups at a time. The overall 16-mer test gave p = 0.100, so these follow-up comparisons are exploratory. Use the adjusted p-values to interpret the three pairwise tests.

In [32]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$kmer16_settlement_distance_path" \
    --m-metadata-file "$settlement_metadata_path" \
    --m-metadata-column Settlement_Type \
    --p-pairwise \
    --o-visualization "$kmer16_pairwise_permanova_path"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/kmer16_bray_curtis_settlement_pairwise_permanova.qzv


In [ ]:
Visualization.load(kmer16_pairwise_permanova_path)

In [33]:
! qiime diversity beta-group-significance \
    --i-distance-matrix "$asv_settlement_distance_path" \
    --m-metadata-file "$settlement_metadata_path" \
    --m-metadata-column Settlement_Type \
    --p-pairwise \
    --o-visualization "$asv_pairwise_permanova_path"

Saved Visualization to: ../data/processed/08_beta_diversity_settlement/asv_bray_curtis_settlement_pairwise_permanova.qzv


In [ ]:
Visualization.load(asv_pairwise_permanova_path)

### Pairwise PERMANOVA by settlement type

| Settlement pair | 16-mer q | Whole-ASV q |
|---|---:|---:|
| Rural vs. Suburban | 0.159 | **0.039** |
| Rural vs. Urban | 0.662 | 0.773 |
| Suburban vs. Urban | 0.054 | **0.027** |

After correction for multiple comparisons, both pairs involving Suburban samples met the q < 0.05 threshold in the whole-ASV analysis. No 16-mer pair met that threshold.

## Discussion

Evidence for an association between settlement type and fungal community composition varied with how we represented the ITS1 sequences. The whole-ASV analysis suggested that comparisons involving Suburban samples may be worth following up, while the primary 16-mer analysis provided weaker evidence. Because both analyses used the same samples and Bray–Curtis distances, this comparison highlights how the choice of features can affect the statistical conclusion.

This also shows that visual clustering is a limited guide to statistical significance. The PCoA scatter plots in notebook 07 showed no clear clusters by settlement type on the first two axes, even though the whole-ASV PERMANOVA detected an overall association using the full distance matrix.